# Member C (Colab Setup)
- Use a T4 GPU runtime.
- Add `GH_TOKEN`, `LANGFUSE_PK`, `LANGFUSE_SK` in Colab **Secrets**.
- Upload `secret_test_questions.jsonl` to the repository root on Saturday (never commit it).


In [ ]:
import os, base64
from google.colab import userdata, drive

gh_token = userdata.get("GH_TOKEN")
pk = userdata.get("LANGFUSE_PK")
sk = userdata.get("LANGFUSE_SK")

!git clone -b c-monitoring https://{gh_token}@github.com/shayanop/QanoonAI.git
%cd QanoonAI
!pip install -q -r requirements.txt opentelemetry-sdk opentelemetry-exporter-otlp

drive.mount("/content/drive")
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = "Authorization=Basic%20" + base64.b64encode(f"{pk}:{sk}".encode()).decode()

def save():
    !mkdir -p /content/drive/MyDrive/G10/C && cp -r results traces /content/drive/MyDrive/G10/C/


In [ ]:
!python -m harness.smoke --group G10 --memory M1 --adapter A0 --accept
save()

In [ ]:
!python -m harness.evaluate --group G10 --memory M0 --adapter A0 --split dev --all-seeds
save()

In [ ]:
!python -m harness.evaluate --group G10 --memory M0 --adapter A0 --split test --questions secret_test_questions.jsonl --all-seeds
save()

In [ ]:
import csv
for seed in (13, 42, 2026):
    fp = f"results/G10_M0+A0_dev_s{seed}.csv"
    try:
        for r in csv.DictReader(open(fp, encoding="utf-8")):
            if r["correct"] == "0" or r["ungrounded_cites"] not in ("", "0"):
                print(r["config"], r["seed"], r["session_id"], r["turn"], r["family"],
                      "ungrounded=" + r["ungrounded_cites"],
                      "abst_exp=" + r["abstain_expected"], "abst=" + r["abstained"],
                      "parse=" + r["parse_ok"], "memrec=" + r["memory_recall"] )
    except FileNotFoundError:
        print("Missing:", fp)


In [ ]:
!python -m harness.summarize --group G10 --split dev
!python -m harness.summarize --group G10 --split dev --latex > report/tables/table_2x2_dev.tex
!python dashboard/make_dashboard.py --group G10 --splits dev test
!python -m harness.trace_export --group G10
